# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbhutto256/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
# Setup: query the hosted warehouse without downloading the full dataset.
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy

import os, getpass, duckdb, pandas as pd, numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

# Safe fallback: interactive prompt; the token is never written into the notebook.
HF_TOKEN = HF_TOKEN or getpass.getpass("Enter your Hugging Face READ token (hidden): ")

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Connected to the March 2026 partition.")


Connected to the March 2026 partition.


## 1. Unit of analysis + time window

### Contract answer 1 — What one row means
For the **raw warehouse fact**, one row means one **client × content item × report date**. For my lane's decision frame, I aggregate those daily rows to **one row per client × content item for the March decision point**.

### Contract answer 2 — Which table(s) I use
I use `fact_content_daily_performance` for daily Google Search Console performance. This notebook's five-feature frame uses the daily fact only, so the contract stays simple and avoids unnecessary joins.

### Contract answer 3 — Time window
I develop on **March 2026**. The decision point is **2026-03-15**: features use March 1–15, while the outcome uses March 16–31. I use `gsc_data_available IS TRUE` so unavailable GSC history is not treated as zero performance.

### Contract answer 4 — What I predict/rank
I use a simple **decline proxy**: `1` when second-half March impressions are more than 20% below first-half March impressions, otherwise `0`. This is decision-support for pages whose search visibility weakened after the decision point; it is not a claim about Google's ranking algorithm.

### Contract answer 5 — What I deliberately exclude
I deliberately exclude **future/outcome-window measurements** from the honest feature set, including second-half impressions, second-half clicks, and any feature calculated from the outcome window. I also exclude client/content IDs from model features because they are pseudonymous grouping keys, not meaningful signals.


In [13]:
# Connection/schema sanity check (not one of the three verification queries).
required = {
    "report_date", "client_hash_id", "content_hash_id",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "gsc_data_available",
}
schema = con.sql(f"DESCRIBE SELECT * FROM {DAILY} LIMIT 0").df()
available = set(schema["column_name"])
missing = sorted(required - available)
if missing:
    raise ValueError(f"Required warehouse columns are missing: {missing}")

print("Required columns found:", ", ".join(sorted(required)))


Required columns found: client_hash_id, content_hash_id, gsc_avg_position, gsc_clicks, gsc_data_available, gsc_impressions, report_date


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Features** | `first_half_impressions`, `first_half_clicks`, `first_half_avg_position`, `first_half_active_days`, `first_half_ctr_pct` | All are calculated only from March 1–15, so they are knowable at the March 15 decision moment. |
| **Label / proxy** | `decline_proxy` | Defined from March 16–31, after the decision moment. |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_available` | Used for grouping, grain checks, filtering, and client-aware analysis; IDs are not model features. |
| **Excluded** | second-half metrics, `future_trend_pct`, future-window rates, raw IDs as predictors | They are future information or meaningless identifiers. `future_trend_pct` is introduced only for the leakage demonstration, then removed. |

The GSC flag is three-valued in the warehouse, so the contract uses **`gsc_data_available IS TRUE`**. I do not replace unavailable GSC rows with zero.

## 3. Verify it with exactly three verification queries

The three required verification queries are: **grain**, **March row count/date span**, and **availability with `IS TRUE`**.


### Verification query 1 — Grain

The documented raw fact grain is `report_date × client_hash_id × content_hash_id`. An empty result means no duplicate raw-grain keys were observed in the March partition.


In [14]:
# VERIFICATION QUERY 1 — raw fact grain
q1 = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_rows
FROM {DAILY}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
ORDER BY duplicate_rows DESC
LIMIT 10
""").df()

print("Duplicate grain keys returned:", len(q1))
display(q1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain keys returned: 0


,report_date,client_hash_id,content_hash_id,duplicate_rows


### Verification query 2 — Row count and date span

This measures the March partition actually used for development.

In [15]:
# VERIFICATION QUERY 2 — March row count and date span
q2 = con.sql(f"""
SELECT COUNT(*) AS row_count,
       MIN(report_date) AS min_report_date,
       MAX(report_date) AS max_report_date
FROM {DAILY}
""").df()

display(q2)


,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


### Verification query 3 — GSC availability

This is the required availability check. The comparison deliberately uses **`IS TRUE`** so NULL is not silently treated as available.

In [16]:
# VERIFICATION QUERY 3 — availability, using IS TRUE
q3 = con.sql(f"""
SELECT
    COUNT(*) AS all_march_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS not_gsc_available_or_null_rows
FROM {DAILY}
""").df()

display(q3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_march_rows,gsc_available_rows,not_gsc_available_or_null_rows
0,9841378,3611061,6230317


## 3b. Five-feature frame

Every feature is computed from **March 1–15 only**, so each is available at the decision moment.

1. **`first_half_impressions`** — knowable because GSC impressions were observed through March 15.
2. **`first_half_clicks`** — knowable because GSC clicks were observed through March 15.
3. **`first_half_avg_position`** — knowable because position is calculated only from observations through March 15.
4. **`first_half_active_days`** — knowable because it counts March 1–15 days with at least one GSC impression.
5. **`first_half_ctr_pct`** — knowable because it uses only first-half clicks and impressions.

The outcome is separate: `decline_proxy = 1` when second-half impressions are below 80% of first-half impressions. Pages with no first-half impressions are excluded because a percentage change would not be meaningful.

In [17]:
# Build the five-feature frame + proxy label.
feature_sql = f"""
WITH daily AS (
    SELECT report_date, client_hash_id, content_hash_id,
           COALESCE(gsc_impressions, 0) AS gsc_impressions,
           COALESCE(gsc_clicks, 0) AS gsc_clicks,
           NULLIF(gsc_avg_position, 0) AS gsc_avg_position
    FROM {DAILY}
    WHERE gsc_data_available IS TRUE
),
agg AS (
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                    THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
           SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                    THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
           AVG(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                    THEN gsc_avg_position END) AS first_half_avg_position,
           COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                    AND gsc_impressions > 0 THEN report_date END) AS first_half_active_days,
           SUM(CASE WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                    THEN gsc_impressions ELSE 0 END) AS second_half_impressions
    FROM daily
    GROUP BY 1, 2
)
SELECT client_hash_id, content_hash_id,
       first_half_impressions,
       first_half_clicks,
       first_half_avg_position,
       first_half_active_days,
       100.0 * first_half_clicks / NULLIF(first_half_impressions, 0) AS first_half_ctr_pct,
       second_half_impressions,
       CASE WHEN second_half_impressions < 0.8 * first_half_impressions THEN 1 ELSE 0 END AS decline_proxy
FROM agg
WHERE first_half_impressions > 0
  AND second_half_impressions IS NOT NULL
"""

feature_frame = con.sql(feature_sql).df()

feature_cols = [
    "first_half_impressions", "first_half_clicks",
    "first_half_avg_position", "first_half_active_days",
    "first_half_ctr_pct"
]

print(f"Feature-frame rows: {len(feature_frame):,}")
print("Five honest features:", feature_cols)
display(feature_frame[["client_hash_id", "content_hash_id"] + feature_cols + ["decline_proxy"]].head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 151,981
Five honest features: ['first_half_impressions', 'first_half_clicks', 'first_half_avg_position', 'first_half_active_days', 'first_half_ctr_pct']


,client_hash_id,content_hash_id,first_half_impressions,first_half_clicks,first_half_avg_position,first_half_active_days,first_half_ctr_pct,decline_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,15,0.143781,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,4.185913,15,0.000000,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,15,0.080972,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,15,0.327869,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,15,0.416667,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,131.0,0.0,9.284735,15,0.000000,1
6,client_73cda7b4e4f265ea,content_1f380a642aed423b,44.0,1.0,10.734848,15,2.272727,0
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,172.0,0.0,7.602850,15,0.000000,0
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,3104.0,16.0,5.536679,15,0.515464,0
9,client_73cda7b4e4f265ea,content_20403327d8d9374c,1294.0,5.0,6.902681,15,0.386399,0


## 3c. The trap — deliberately leak the label-derived signal

For the leakage demonstration, I add exactly **one** deliberately unsafe column:

`future_trend_pct = (second_half_impressions - first_half_impressions) / first_half_impressions × 100`

The label is defined directly from that future movement (`future_trend_pct < -20`). Therefore this column is label-derived and cannot be known at the March 15 decision moment.

I compare a quick model using the five honest features against the same model after adding this one leaked column. Then I delete the leaked column and keep the honest score.

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

model_df = feature_frame.copy()
model_df["future_trend_pct"] = (
    (model_df["second_half_impressions"] - model_df["first_half_impressions"])
    / model_df["first_half_impressions"]
) * 100.0

X_train, X_test, y_train, y_test = train_test_split(
    model_df[feature_cols], model_df["decline_proxy"],
    test_size=0.25, random_state=42, stratify=model_df["decline_proxy"]
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000, random_state=42)
)
honest_model.fit(X_train, y_train)
honest_prob = honest_model.predict_proba(X_test)[:, 1]
honest_pred = (honest_prob >= 0.5).astype(int)

leak_cols = feature_cols + ["future_trend_pct"]
LX_train, LX_test, Ly_train, Ly_test = train_test_split(
    model_df[leak_cols], model_df["decline_proxy"],
    test_size=0.25, random_state=42, stratify=model_df["decline_proxy"]
)

leak_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000, random_state=42)
)
leak_model.fit(LX_train, Ly_train)
leak_prob = leak_model.predict_proba(LX_test)[:, 1]
leak_pred = (leak_prob >= 0.5).astype(int)

honest_auc = roc_auc_score(y_test, honest_prob)
leak_auc = roc_auc_score(Ly_test, leak_prob)

print(f"Honest model — accuracy: {accuracy_score(y_test, honest_pred):.3f}, ROC-AUC: {honest_auc:.3f}")
print(f"Leaked model — accuracy: {accuracy_score(Ly_test, leak_pred):.3f}, ROC-AUC: {leak_auc:.3f}")
print()
print("LEAK CHECK: future_trend_pct is unsafe because it uses the outcome window.")
print("The honest feature set is retained; the leaked column is removed below.")

honest_feature_frame = feature_frame.drop(columns=["second_half_impressions"])
assert "future_trend_pct" not in honest_feature_frame.columns
print("Retained columns:", honest_feature_frame.columns.tolist())


Honest model — accuracy: 0.673, ROC-AUC: 0.601
Leaked model — accuracy: 1.000, ROC-AUC: 1.000

LEAK CHECK: future_trend_pct is unsafe because it uses the outcome window.
The honest feature set is retained; the leaked column is removed below.
Retained columns: ['client_hash_id', 'content_hash_id', 'first_half_impressions', 'first_half_clicks', 'first_half_avg_position', 'first_half_active_days', 'first_half_ctr_pct', 'decline_proxy']


## 4. Data limits

**Named limitation:** the warehouse is an **unbalanced panel**. Different clients have different tracking-history start dates, so a missing daily observation cannot automatically be interpreted as zero performance or as a comparable amount of history across clients. This March slice also represents only pages with `gsc_data_available IS TRUE`, so the results describe the measurable GSC-covered portion of the warehouse, not every content item.

A second limitation is that the decline proxy is deliberately simple: it compares two half-month windows. It is useful for demonstrating a clean decision point and leakage discipline, but it is not a causal explanation of why a page changed and should be treated as **decision-support**, not a prediction of Google's ranking algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.